In [1]:
import numpy as np
%matplotlib qt
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.optimize import curve_fit

In [4]:
# PLOTTING ABSORPTION
path1 = "./data/feature_fitting_test/reflectance_no_grating.txt"
path2 = "./data/feature_fitting_test/reflectance_test_grating.txt"

background = np.loadtxt(path1)   # shape (N,2)
grating = np.loadtxt(path2)

plt.figure(figsize=(10,6))
plt.plot(grating[:,0][0:-200], (grating[:,1][0:-200] / np.max(grating[:,1][0:-200])) , color='k', label="Test Grating")
plt.plot(background[:,0][0:-200], (background[:,1][0:-200] / np.max(background[:,1][0:-200])) , color='r', linestyle="--", label="Substrate Only")
plt.xlabel("Wavelength (um)", fontsize=20)
plt.ylabel("Normalized Reflectance (%)", fontsize=20)
plt.title("6H SiC FTIR", fontsize=20)
plt.legend(fontsize=20)
plt.grid(alpha=0.8)
plt.show()

plt.figure(figsize=(10,6))
plt.plot(grating[:,0][0:-200], -1 * ((grating[:,1][0:-200] / np.max(grating[:,1][0:-200])) - (background[:,1][0:-200] / np.max(background[:,1][0:-200]))), color='k', label="Grating Absorption")
plt.xlabel("Wavelength (um)", fontsize=20)
plt.ylabel("Normalized Absorption (%)", fontsize=20)
plt.title("6H SiC Grating Absorption", fontsize=20)
plt.legend(fontsize=20)
plt.grid(alpha=0.8)
plt.show()

curve_x = grating[:,0][0:-200]
curve_y = -1 * ((grating[:,1][0:-200] / np.max(grating[:,1][0:-200])) - (background[:,1][0:-200] / np.max(background[:,1][0:-200])))

def upsample_linear(x, y, n_between):
    x = np.asarray(x)
    y = np.asarray(y)
    total_points = (len(x) - 1) * (n_between + 1) + 1
    x_new = np.linspace(x[0], x[-1], total_points)
    y_new = np.interp(x_new, x, y)
    return x_new, y_new

curve_x, curve_y = upsample_linear(curve_x, curve_y, n_between=10)

In [3]:
PEAK_THRESHOLD = 0.2        # minimum y-value AFTER baseline subtraction
WINDOW_PTS = 20             # points on each side for fitting
N_GAMMA_PLOT = 10           # plot ±N·gamma around each peak

def lorentzian(x, A, x0, gamma):
    """Lorentzian with HWHM = gamma (no offset)"""
    return A * (gamma**2 / ((x - x0)**2 + gamma**2))

def analyze_spectrum(x, y, threshold, window_pts):
    peaks, _ = find_peaks(y, height=threshold)
    results = []

    for p in peaks:
        left = max(0, p - window_pts)
        right = min(len(x), p + window_pts)

        x_fit = x[left:right]
        y_fit = y[left:right]

        # Initial guesses
        A0 = y[p]
        x0_0 = x[p]
        gamma_0 = (x_fit[-1] - x_fit[0]) / 10

        p0 = [A0, x0_0, gamma_0]

        popt, pcov = curve_fit(
            lorentzian,
            x_fit,
            y_fit,
            p0=p0,
            bounds=(
                [0, x_fit[0], 0],
                [np.inf, x_fit[-1], np.inf]
            )
        )

        results.append({
            "peak_index": p,
            "left_idx": left,
            "right_idx": right,
            "A": popt[0],
            "x0": popt[1],
            "gamma": popt[2],
            "popt": popt,
        })

    return results

results = analyze_spectrum(
    curve_x,
    curve_y,
    threshold=PEAK_THRESHOLD,
    window_pts=WINDOW_PTS
)

plt.figure(figsize=(10, 6))
plt.plot(curve_x, curve_y, color="k", label="Baseline-subtracted data")

for r in results:
    # Plot full Lorentzian decay (± N·gamma)
    x_dense = np.linspace(
        max(curve_x[0], r["x0"] - N_GAMMA_PLOT * r["gamma"]),
        min(curve_x[-1], r["x0"] + N_GAMMA_PLOT * r["gamma"]),
        800
    )

    plt.plot(
        x_dense,
        lorentzian(x_dense, *r["popt"]),
        "--",
        linewidth=2,
        label=f"x₀={r['x0']:.3f}, γ={r['gamma']:.3e}"
    )

    # Window visualization
    x_left = curve_x[r["left_idx"]]
    x_right = curve_x[r["right_idx"] - 1]

    plt.axvline(x_left, color="gray", linestyle=":", alpha=0.5)
    plt.axvline(x_right, color="gray", linestyle=":", alpha=0.5)
    plt.axvspan(x_left, x_right, color="gray", alpha=0.08)

    # Peak center
    plt.axvline(r["x0"], color="red", linestyle="--", alpha=0.6)

plt.xlabel("Wavelength (µm)", fontsize=20)
plt.ylabel("Absorption (baseline subtracted)", fontsize=20)
plt.legend(fontsize=20)
plt.title("Absorption Features With Fits", fontsize=20)
plt.grid(alpha=0.8)
plt.show()

print("Detected Lorentzian Peaks:")
print("-" * 45)

for i, r in enumerate(results, 1):
    print(
        f"Peak {i}: "
        f"x0 = {r['x0']:.6f} µm, "
        f"gamma = {r['gamma']:.6e}"
    )

Detected Lorentzian Peaks:
---------------------------------------------
Peak 1: x0 = 10.542402 µm, gamma = 3.312232e-02
Peak 2: x0 = 10.571626 µm, gamma = 5.532287e-02
Peak 3: x0 = 11.084790 µm, gamma = 1.852873e-02
